# 🌸 Kira: Manga Upscale & Kindle Adaptation Pipeline

**Kira** é um pipeline automatizado para melhorar a resolução de Mangás utilizando **Real-ESRGAN** (otimizado para artes de anime/mangá via GPU no Google Colab) e adaptá-los perfeitamente para e-readers **Amazon Kindle** usando **KCC (Kindle Comic Converter)**.

---

### 📁 Passo 1: Montar o Google Drive
Execute a célula abaixo para conectar a sua conta do Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive montado com sucesso!")

In [ ]:
# Bootstrap único do Kira e das dependências do Colab
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Wather17/kira.git"
KIRA_ROOT = Path("/content/kira")
DRIVE_REPO = Path("/content/drive/MyDrive/kira")

def run_checked(command, step):
    print(f"▶️ {step}...")
    try:
        return subprocess.run(command, check=True, capture_output=True, text=True)
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        detail = getattr(exc, "stderr", "") or getattr(exc, "stdout", "") or str(exc)
        raise RuntimeError(f"{step} falhou: {detail.strip()}") from exc

def validate_checkout(path):
    required = (path / "pyproject.toml", path / "kira")
    if not path.is_dir() or not all(item.exists() for item in required) or not (path / ".git").exists():
        raise RuntimeError(f"Checkout inválido ou não verificável: {path}")
    result = run_checked(["git", "-C", str(path), "rev-parse", "--short", "HEAD"], "Validar commit do Kira")
    return result.stdout.strip()

run_checked(["apt-get", "update", "-qq"], "Atualizar índice do apt")
run_checked(["apt-get", "install", "-y", "-qq", "p7zip-full", "unrar"], "Instalar ferramentas de arquivo")

if KIRA_ROOT.exists():
    checkout_source = "checkout da sessão (/content/kira)"
elif DRIVE_REPO.exists():
    run_checked(["git", "clone", "--local", str(DRIVE_REPO), str(KIRA_ROOT)], "Copiar checkout validado do Google Drive")
    checkout_source = "checkout do Google Drive"
else:
    run_checked(["git", "clone", "--depth", "1", REPO_URL, str(KIRA_ROOT)], "Clonar Kira do repositório oficial")
    checkout_source = REPO_URL

commit = validate_checkout(KIRA_ROOT)
if str(KIRA_ROOT) not in sys.path:
    sys.path.insert(0, str(KIRA_ROOT))

# Real-ESRGAN depende do BasicSR; o projeto fixa uma revisão compatível com Python 3.13.
run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", str(KIRA_ROOT)], "Instalar Kira e dependências runtime")
run_checked([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ciromattia/kcc.git"], "Instalar KCC")
run_checked([sys.executable, "-m", "pip", "install", "-q", "jedi>=0.16"], "Instalar Jedi exigido pelo IPython do Colab")
run_checked([sys.executable, "-m", "pip", "check"], "Validar dependências Python")
importlib.import_module("kira.pipeline")
kcc_bin = shutil.which("kcc-c2e") or shutil.which("kcc")
if not kcc_bin:
    raise RuntimeError("KCC não foi encontrado após a instalação.")
kcc_version = run_checked([kcc_bin, "--help"], "Validar versão do KCC")
version_line = (kcc_version.stdout or kcc_version.stderr).splitlines()[:1]
print(f"✅ Kira pronto: {checkout_source} ({commit})")
print(f"✅ KCC pronto: {version_line[0] if version_line else kcc_bin}")

In [ ]:
#@title ⚙️ Configurações do Pipeline do Kira { display-mode: "form" }

import sys, os
if os.path.exists("/content/kira") and "/content/kira" not in sys.path:
    sys.path.insert(0, "/content/kira")

#@markdown **Diretórios no Google Drive:**
Manga_Input_Folder = "MyDrive/Manga_Inputs" #@param {type:"string"}
Kindle_Output_Folder = "MyDrive/Kindle_Outputs" #@param {type:"string"}

#@markdown **Configurações do Real-ESRGAN (Upscale IA):**
RealESRGAN_Model = "RealESRGAN_x4plus_anime_6B" #@param ["RealESRGAN_x4plus_anime_6B", "realesr-animevideov3", "RealESRGAN_x4plus"]
GPU_Tile_Size = 400 #@param {type:"integer"}
Concurrent_Workers = 2 #@param {type:"integer"}
Grayscale_EInk = False #@param {type:"boolean"}
Max_Dimension_Px = 2400 #@param {type:"integer"}

#@markdown **Configurações do Kindle (KCC):**
Kindle_Device = "K11" #@param ["KPW5", "KPW34", "KPW", "KO", "KS", "K11", "KV", "K34", "K57", "OTHER", "KPW3", "K345"]
Output_Format = "EPUB" #@param ["EPUB", "CBZ", "KFX"]
Cropping_Mode = 0 #@param [0, 1, 2]
#@markdown Cropping: `0` preserva a página; `1` remove margens; `2` remove margens e numeração.
#@markdown `AZW3` e `MOBI` são formatos legados da CLI e resultam em `EPUB`; não são outputs independentes.
Gamma_Correction = 1.0 #@param {type:"number"}
Keep_Upscaled_CBZ = True #@param {type:"boolean"}

from kira.pipeline import MangaPipeline
from kira.cli import _print_summary
from pathlib import Path

print("▶️ Iniciando processamento...")

# Inicializar Pipeline com Concorrência
pipeline = MangaPipeline(
    model_name=RealESRGAN_Model,
    tile=GPU_Tile_Size,
    grayscale=Grayscale_EInk,
    max_dimension=Max_Dimension_Px,
    kindle_profile=Kindle_Device,
    output_format=Output_Format,
    gamma=Gamma_Correction,
    cropping=Cropping_Mode,
    keep_upscaled_cbz=Keep_Upscaled_CBZ,
    workers=Concurrent_Workers
)

# Processar diretório ou arquivo
results = pipeline.process_directory(Manga_Input_Folder, Kindle_Output_Folder)

print("\n🎉 Processamento concluído com sucesso!")
_print_summary(results)
